In [ ]:
import os
os.chdir("../../")

## Chargement des données

In [16]:
import pandas as pd

X = pd.read_csv("Data/dpe_explicative.csv", index_col=0)
Y = pd.read_csv("Data/dpe_cible.csv", index_col=0)
Y = Y.etiquette_dpe

print(X.shape)
print(Y.shape)

(58011, 19)
(58011,)


In [17]:
X.info()

<class 'pandas.DataFrame'>
Index: 58011 entries, 2420N3326336G to 2620E2590403W
Data columns (total 19 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   qualite_isolation_menuiseries      58011 non-null  str    
 1   qualite_isolation_plancher_bas     58011 non-null  str    
 2   qualite_isolation_enveloppe        58011 non-null  str    
 3   type_energie_principale_ecs        58011 non-null  str    
 4   classe_inertie_batiment            58011 non-null  str    
 5   type_batiment                      58011 non-null  str    
 6   periode_construction               58011 non-null  str    
 7   zone_climatique                    58011 non-null  str    
 8   qualite_isolation_murs             58011 non-null  str    
 9   classe_altitude                    58011 non-null  str    
 10  type_energie_principale_chauffage  58011 non-null  str    
 11  indicateur_confort_ete             58011 non-null 

## Préparation des données

In [22]:
# Traitement des variables
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector

variables_numeriques = X.select_dtypes(include=float)
variables_categorielles = X.select_dtypes(exclude=float)

# Pipeline quanti
pipeline_numerique = Pipeline([
    ('standardisation', StandardScaler()) #On a des échelles bien différentes, par précaussion on standardise
])

# Pipeline quali
pipeline_categoriel = Pipeline([
    ('encodage', OneHotEncoder(handle_unknown='ignore'))
])

# Préprocesseur global
preprocesseur = ColumnTransformer([
    ('quanti', pipeline_numerique, make_column_selector(dtype_include="float")),
    ('quali', pipeline_categoriel, make_column_selector(dtype_include="str"))
])

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,Y,test_size=0.20,random_state=38,stratify=Y)

print("Taille apprentissage :", X_train.shape)
print("Taille test :", X_test.shape)

Taille apprentissage : (46408, 19)
Taille test : (11603, 19)


In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Création du modèle
pipeline = Pipeline([
    ('pretraitement', preprocesseur),
    ('modele', LogisticRegression(
        max_iter=5000,
        solver='lbfgs'
    ))
])

# Hyperparamètres à tester
parametres = {
    'modele__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'modele__class_weight': [None, 'balanced']
}

# Validation croisée stratifiée
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Recherche des meilleurs paramètres
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=parametres,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

# Entraînement
grid_search.fit(X_train, y_train)

# Meilleur modèle
meilleur_modele = grid_search.best_estimator_

print("Meilleurs paramètres :", grid_search.best_params_)
print("Meilleur F1 macro en validation :", grid_search.best_score_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Meilleurs paramètres : {'modele__C': 10, 'modele__class_weight': 'balanced'}
Meilleur F1 macro en validation : 0.4267227265720301


In [32]:
from sklearn.metrics import accuracy_score, f1_score,classification_report

# Prédictions
y_pred = grid_search.predict(X_test)

# Indicateurs de performance
accuracy = accuracy_score(y_test, y_pred)

f1_macro = f1_score(
    y_test, y_pred,
    average='macro'
)

print("===== PERFORMANCE DU MODÈLE =====")

print(f"Accuracy : {accuracy:.4f}")
print(f"Balanced Accuracy : {balanced_accuracy:.4f}")
print(f"F1-score macro : {f1_macro:.4f}")

print("\n===== RAPPORT DE CLASSIFICATION =====")

print(classification_report(
    y_test,
    y_pred,
    labels=['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    zero_division=0
))


===== PERFORMANCE DU MODÈLE =====
Accuracy : 0.5221
Balanced Accuracy : 0.5047
F1-score macro : 0.4185

===== RAPPORT DE CLASSIFICATION =====
              precision    recall  f1-score   support

           A       0.79      0.86      0.82      2050
           B       0.42      0.56      0.48      1864
           C       0.80      0.45      0.58      4853
           D       0.36      0.40      0.38      1733
           E       0.18      0.30      0.23       682
           F       0.15      0.29      0.20       295
           G       0.15      0.67      0.25       126

    accuracy                           0.52     11603
   macro avg       0.41      0.50      0.42     11603
weighted avg       0.61      0.52      0.54     11603



In [36]:
import numpy as np

np.unique(y_pred, return_counts=True)

(array(['A', 'B', 'C', 'D', 'E', 'F', 'G'], dtype=object),
 array([2219, 2504, 2725, 1887, 1124,  579,  565]))

In [37]:
np.unique(y_test, return_counts=True)

(array(['A', 'B', 'C', 'D', 'E', 'F', 'G'], dtype=object),
 array([2050, 1864, 4853, 1733,  682,  295,  126]))